In [1]:
# Nigeria LPG Facility Probability Model & Allocation Parameters
#
# This notebook:
# 1. Aligns population, urban, income, and friction rasters.
# 2. Computes urban/rural facility counts and LPG user estimates.
# 3. Builds a cell‑level dataset (presence/absence + predictors).
# 4. Fits logistic regression models for resellers and filling stations.
# 5. Calculates median nearest‑neighbour distances (for anti‑clustering).
# 6. Exports the models and parameters for use in other countries.

# %%
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import reproject, Resampling
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pickle
import random
from scipy.spatial import cKDTree

# ============================
# PARAMETERS
# ============================
DATA_DIR = "SSA/nige_dataset"
GPKG_NAME = "full_lpg_chain_nig_3857.gpkg"
POP_TIF   = "population.tif"
URBAN_TIF = "urban.tif"
INCOME_TIF = "income_nigeria.tif"
FRICTION_TIF = "friction_moto.tif"          # minutes per meter (motorized)

URBAN_SHARE = 0.25468      # ONSTOVE urban LPG adoption share
RURAL_SHARE = 0.027905
URBAN_THRESHOLD = 20       # value in urban.tif above which a cell is "urban"
TARGET_CRS = "EPSG:3857"

# -----------------------------------------------------------------
# 1. LOAD VECTOR DATA
# -----------------------------------------------------------------
gpkg_path = os.path.join(DATA_DIR, GPKG_NAME)
resell = gpd.read_file(gpkg_path, layer='resell')
filling = gpd.read_file(gpkg_path, layer='filling')

if resell.crs != TARGET_CRS:
    resell = resell.to_crs(TARGET_CRS)
if filling.crs != TARGET_CRS:
    filling = filling.to_crs(TARGET_CRS)

print(f"Resellers: {len(resell)}  Filling: {len(filling)}")

# -----------------------------------------------------------------
# 2. LOAD & ALIGN RASTERS
# -----------------------------------------------------------------
pop_path   = os.path.join(DATA_DIR, POP_TIF)
urb_path   = os.path.join(DATA_DIR, URBAN_TIF)
inc_path   = os.path.join(DATA_DIR, INCOME_TIF)
fric_path  = os.path.join(DATA_DIR, FRICTION_TIF)

# Reference raster (population)
with rasterio.open(pop_path) as ref:
    ref_shape = ref.shape
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_nodata = ref.nodata
    pop_data = ref.read(1).astype(float)
    if ref_nodata is not None:
        pop_data[pop_data == ref_nodata] = np.nan

def align_raster(src_path, ref_shape, ref_transform, ref_crs):
    """Resample a raster to the reference grid; return array with nodata->NaN."""
    with rasterio.open(src_path) as src:
        resampled = np.empty(ref_shape, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=resampled,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.bilinear
        )
        src_nodata = src.nodata
        if src_nodata is not None:
            resampled[resampled == src_nodata] = np.nan
        return resampled

urban_data = align_raster(urb_path, ref_shape, ref_transform, ref_crs)
income_data = align_raster(inc_path, ref_shape, ref_transform, ref_crs)
friction_data = align_raster(fric_path, ref_shape, ref_transform, ref_crs)

# Urban / rural mask
urban_mask = (urban_data >= URBAN_THRESHOLD) & (~np.isnan(urban_data))
rural_mask = (urban_data < URBAN_THRESHOLD) & (~np.isnan(urban_data))
print(f"All rasters aligned to shape {ref_shape}")

# -----------------------------------------------------------------
# 3. BASIC COUNTS & LPG USER ESTIMATES
# -----------------------------------------------------------------
def point_in_mask(gdf, mask, transform):
    coords = [(p.x, p.y) for p in gdf.geometry]
    rows, cols = zip(*[rowcol(transform, x, y) for x, y in coords])
    rows, cols = np.array(rows), np.array(cols)
    valid = (rows >= 0) & (rows < mask.shape[0]) & (cols >= 0) & (cols < mask.shape[1])
    in_mask = np.zeros(len(gdf), dtype=bool)
    if np.any(valid):
        in_mask[valid] = mask[rows[valid], cols[valid]]
    return in_mask

res_urban = point_in_mask(resell, urban_mask, ref_transform)
res_rural = point_in_mask(resell, rural_mask, ref_transform)
fill_urban = point_in_mask(filling, urban_mask, ref_transform)
fill_rural = point_in_mask(filling, rural_mask, ref_transform)

n_res_urban = res_urban.sum()
n_res_rural = res_rural.sum()
n_fill_urban = fill_urban.sum()
n_fill_rural = fill_rural.sum()

urban_pop = np.nansum(pop_data[urban_mask])
rural_pop = np.nansum(pop_data[rural_mask])
urban_lpg = URBAN_SHARE * urban_pop
rural_lpg = RURAL_SHARE * rural_pop
total_lpg = urban_lpg + rural_lpg

res_per_1k_urban = n_res_urban / urban_lpg * 1000 if urban_lpg else np.nan
res_per_1k_rural = n_res_rural / rural_lpg * 1000 if rural_lpg else np.nan
fill_per_1k_urban = n_fill_urban / urban_lpg * 1000 if urban_lpg else np.nan
fill_per_1k_rural = n_fill_rural / rural_lpg * 1000 if rural_lpg else np.nan

print("\n--- Facility counts ---")
print(f"Resellers: urban {n_res_urban} ({100*n_res_urban/len(resell):.1f}%)  rural {n_res_rural} ({100*n_res_rural/len(resell):.1f}%)")
print(f"Filling:   urban {n_fill_urban} ({100*n_fill_urban/len(filling):.1f}%)  rural {n_fill_rural} ({100*n_fill_rural/len(filling):.1f}%)")
print("\n--- LPG users (millions) ---")
print(f"Urban: {urban_lpg/1e6:.2f}  Rural: {rural_lpg/1e6:.2f}  Total: {total_lpg/1e6:.2f}")
print("\n--- Facilities per 1000 LPG users ---")
print(f"Resellers: urban {res_per_1k_urban:.3f}  rural {res_per_1k_rural:.3f}")
print(f"Filling:   urban {fill_per_1k_urban:.3f}  rural {fill_per_1k_rural:.3f}")

# -----------------------------------------------------------------
# 4. BUILD CELL‑LEVEL TRAINING DATASET
# -----------------------------------------------------------------
# Create presence rasters (1 if at least one facility in cell)
def presence_raster(gdf, shape, transform):
    pres = np.zeros(shape, dtype=np.uint8)
    coords = [(p.x, p.y) for p in gdf.geometry]
    for x, y in coords:
        r, c = rowcol(transform, x, y)
        if 0 <= r < shape[0] and 0 <= c < shape[1]:
            pres[r, c] = 1
    return pres

res_pres = presence_raster(resell, ref_shape, ref_transform)
fill_pres = presence_raster(filling, ref_shape, ref_transform)

# Define a valid mask (all predictors non-NaN)
valid_cells = (~np.isnan(pop_data)) & (~np.isnan(income_data)) & (~np.isnan(urban_data)) & (~np.isnan(friction_data))
# Also exclude cells with zero population? (logistic can handle, but may add noise)
# Keep all valid cells

# Flatten
pop_flat = pop_data[valid_cells]
income_flat = income_data[valid_cells]
urban_flat = urban_mask[valid_cells].astype(np.uint8)   # 1 if urban, else 0
friction_flat = friction_data[valid_cells]
res_flat = res_pres[valid_cells]
fill_flat = fill_pres[valid_cells]

# Prepare design matrix X
X = np.column_stack([income_flat, pop_flat, urban_flat, friction_flat])
y_res = res_flat
y_fill = fill_flat

# Normalize income and friction (optional, but useful for interpretation)
scaler_income = StandardScaler()
scaler_friction = StandardScaler()
scaler_pop = StandardScaler()

X_norm = X.copy()
X_norm[:,0] = scaler_income.fit_transform(income_flat.reshape(-1,1)).ravel()
X_norm[:,1] = scaler_pop.fit_transform(pop_flat.reshape(-1,1)).ravel()
X_norm[:,3] = scaler_friction.fit_transform(friction_flat.reshape(-1,1)).ravel()
# urban is already binary, no scaling needed

print(f"\nTraining cells: {X_norm.shape[0]}  (reseller presence: {y_res.sum()}, filling presence: {y_fill.sum()})")

# -----------------------------------------------------------------
# 5. FIT LOGISTIC REGRESSION MODELS
# -----------------------------------------------------------------
model_res = LogisticRegression(max_iter=1000, solver='lbfgs')
model_res.fit(X_norm, y_res)

model_fill = LogisticRegression(max_iter=1000, solver='lbfgs')
model_fill.fit(X_norm, y_fill)

# Coefficients
feature_names = ['income_norm', 'population_norm', 'is_urban', 'friction_norm']
coef_df = pd.DataFrame({
    'Reseller_coef': model_res.coef_[0],
    'Filling_coef': model_fill.coef_[0]
}, index=feature_names)
print("\nLogistic regression coefficients (normalized predictors):")
print(coef_df)

# Save models & scalers
os.makedirs('SSA/models', exist_ok=True)
with open('SSA/models/logres_reseller.pkl', 'wb') as f:
    pickle.dump(model_res, f)
with open('SSA/models/logres_filling.pkl', 'wb') as f:
    pickle.dump(model_fill, f)
with open('SSA/models/scaler_income.pkl', 'wb') as f:
    pickle.dump(scaler_income, f)
with open('SSA/models/scaler_population.pkl', 'wb') as f:
    pickle.dump(scaler_pop, f)
with open('SSA/models/scaler_friction.pkl', 'wb') as f:
    pickle.dump(scaler_friction, f)

# -----------------------------------------------------------------
# 6. NEAREST‑NEIGHBOUR STATISTICS
# -----------------------------------------------------------------
def median_nn_distance(gdf, transform, mask=None):
    """Median nearest-neighbour distance (in projected units, meters here) for points
    falling inside mask (if provided)."""
    if mask is not None:
        gdf_sub = gdf[point_in_mask(gdf, mask, transform)]
    else:
        gdf_sub = gdf
    if len(gdf_sub) <= 1:
        return np.nan
    coords = np.array([(p.x, p.y) for p in gdf_sub.geometry])
    tree = cKDTree(coords)
    distances, _ = tree.query(coords, k=2)   # closest is itself, second is nearest neighbour
    nn_dist = distances[:, 1]
    return np.median(nn_dist)

# All points
med_res = median_nn_distance(resell, ref_transform)
med_fill = median_nn_distance(filling, ref_transform)

# By urban/rural
med_res_urb = median_nn_distance(resell, ref_transform, urban_mask)
med_res_rur = median_nn_distance(resell, ref_transform, rural_mask)
med_fill_urb = median_nn_distance(filling, ref_transform, urban_mask)
med_fill_rur = median_nn_distance(filling, ref_transform, rural_mask)

print("\n--- Median nearest‑neighbour distances (meters) ---")
print(f"Resellers  : all {med_res:.0f}   urban {med_res_urb:.0f}   rural {med_res_rur:.0f}")
print(f"Filling    : all {med_fill:.0f}   urban {med_fill_urb:.0f}   rural {med_fill_rur:.0f}")

# Compute overall density for scaling
# Area per cell (assuming square cells in projected CRS, 3857)
dx = abs(ref_transform[0])   # pixel width in meters
area_cell_m2 = dx * dx       # assuming square pixels
area_km2 = (ref_shape[0] * ref_shape[1] * area_cell_m2) / 1e6
density_res = len(resell) / area_km2
density_fill = len(filling) / area_km2
print(f"\nFacility density (per km²): Resellers {density_res:.6f}  Filling {density_fill:.6f}")

# -----------------------------------------------------------------
# 7. SAVE SUMMARY PARAMETERS
# -----------------------------------------------------------------
params = {
    'facilities_per_1000_lpg_users': {
        'reseller_urban': res_per_1k_urban,
        'reseller_rural': res_per_1k_rural,
        'filling_urban': fill_per_1k_urban,
        'filling_rural': fill_per_1k_rural
    },
    'median_nn_distance_m': {
        'reseller_all': med_res, 'reseller_urban': med_res_urb, 'reseller_rural': med_res_rur,
        'filling_all': med_fill, 'filling_urban': med_fill_urb, 'filling_rural': med_fill_rur
    },
    'density_per_km2': {
        'reseller': density_res,
        'filling': density_fill
    },
    'urban_share': URBAN_SHARE,
    'rural_share': RURAL_SHARE,
    'urban_threshold': URBAN_THRESHOLD,
    'reference_crs': TARGET_CRS,
    'pixel_size_m': dx
}

with open('SSA/models/allocation_params.pkl', 'wb') as f:
    pickle.dump(params, f)

# Also export a readable CSV of coefficients
coef_df.to_csv('SSA/models/logistic_coefficients.csv')

print("\nAll models and parameters saved in ./models/")
print("Ready for allocation script.")

Resellers: 2413  Filling: 375
All rasters aligned to shape (1085, 1337)

--- Facility counts ---
Resellers: urban 1959 (81.2%)  rural 454 (18.8%)
Filling:   urban 332 (88.5%)  rural 43 (11.5%)

--- LPG users (millions) ---
Urban: 21.06  Rural: 3.43  Total: 24.49

--- Facilities per 1000 LPG users ---
Resellers: urban 0.093  rural 0.133
Filling:   urban 0.016  rural 0.013

Training cells: 560042  (reseller presence: 1924, filling presence: 349)

Logistic regression coefficients (normalized predictors):
                 Reseller_coef  Filling_coef
income_norm           0.348249      0.652008
population_norm       0.023836      0.017793
is_urban              2.555258      1.311884
friction_norm        -2.538511     -0.918083

--- Median nearest‑neighbour distances (meters) ---
Resellers  : all 803   urban 694   rural 5276
Filling    : all 2109   urban 2084   rural 4469

Facility density (per km²): Resellers 0.001663  Filling 0.000259

All models and parameters saved in ./models/
Ready for